# F1 Team Radio Transcription & Classification Pipeline

Processes all race-only team radio recordings using Whisper `base` model on GPU.
- Downloads MP3s from F1 live timing servers
- Transcribes with OpenAI Whisper
- Classifies into structured event JSON

In [ ]:
# Install dependencies
!pip install -q openai-whisper requests pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 14.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

CUDA available: True
GPU: Tesla T4


## 1. Fetch Data from OpenF1 API

In [ ]:
import requests
import pandas as pd
import os

BASE_URL = "https://api.openf1.org/v1"
os.makedirs("data", exist_ok=True)

# Fetch sessions to identify races
print("Fetching sessions...")
sessions = pd.DataFrame(requests.get(f"{BASE_URL}/sessions", timeout=60).json())
sessions.to_csv("data/sessions.csv", index=False)
print(f"  {len(sessions)} sessions")

# Fetch all team radio
print("Fetching team radio...")
radio = pd.DataFrame(requests.get(f"{BASE_URL}/team_radio", timeout=120).json())
radio.to_csv("data/team_radio.csv", index=False)
print(f"  {len(radio)} total recordings")

# Filter to race sessions only
race_keys = sessions[sessions["session_type"] == "Race"]["session_key"]
race_radio = radio[radio["session_key"].isin(race_keys)].reset_index(drop=True)
print(f"  {len(race_radio)} race-only recordings")

Fetching sessions...
  490 sessions
Fetching team radio...
  16425 total recordings
  6989 race-only recordings


## 2. Load Whisper Model

In [ ]:
import whisper

model = whisper.load_model("base", device="cuda" if torch.cuda.is_available() else "cpu")
print(f"Whisper 'base' model loaded on {next(model.parameters()).device}")

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 127MiB/s]


Whisper 'base' model loaded on cuda:0


## 3. Classification Logic

In [ ]:
import re
import json
import tempfile

# Keyword patterns
PIT_KW = re.compile(r"\b(box|pit|pitting|pit\s*stop|pit\s*lane|pit\s*wall|pit\s*now|box\s*box)\b", re.I)
TIRE_KW = re.compile(r"\b(tyre|tire|soft|medium|hard|inter|wet|compound|graining|degradation|deg|blistering|wear)\b", re.I)
SAFETY_KW = re.compile(r"\b(yellow|safety\s*car|vsc|virtual|red\s*flag|caution|incident|crash|off|barrier|debris)\b", re.I)
PACE_KW = re.compile(r"\b(push|pushing|attack|lift|coast|lift\s*and\s*coast|delta|target|pace|manage|conserve|save)\b", re.I)
DAMAGE_KW = re.compile(r"\b(damage|wing|floor|puncture|broken|loose|endplate)\b", re.I)
MECH_KW = re.compile(r"\b(engine|power\s*unit|gearbox|brake|battery|ers|mgu|overheating|temperature|cooling|hydraulic|steering)\b", re.I)
WEATHER_KW = re.compile(r"\b(rain|wet|dry|drizzle|shower|spray|standing\s*water)\b", re.I)
OVERTAKE_KW = re.compile(r"\b(overtake|pass|passing|move|sent\s*it|inside|outside|late\s*brake)\b", re.I)
DEFEND_KW = re.compile(r"\b(defend|defending|cover|position|hold|protect)\b", re.I)
DRS_KW = re.compile(r"\b(drs|detection|activation)\b", re.I)
TRAFFIC_KW = re.compile(r"\b(traffic|blue\s*flag|backmarker|lapped)\b", re.I)
GAP_KW = re.compile(r"\b(gap|interval|behind|ahead|margin|undercut|overcut)\b", re.I)
POSITIVE_KW = re.compile(r"\b(well\s*done|great|nice|perfect|brilliant|fantastic|good\s*job|p[1-3]|win|winner|podium|congratulations)\b", re.I)

ALL_PATTERNS = [PIT_KW, TIRE_KW, SAFETY_KW, PACE_KW, DAMAGE_KW, MECH_KW,
                WEATHER_KW, OVERTAKE_KW, DEFEND_KW, DRS_KW, TRAFFIC_KW]


def classify_radio(transcript, record):
    text = transcript.strip()
    text_lower = text.lower()

    hits = {
        "pit": bool(PIT_KW.search(text)), "tire": bool(TIRE_KW.search(text)),
        "safety": bool(SAFETY_KW.search(text)), "pace": bool(PACE_KW.search(text)),
        "damage": bool(DAMAGE_KW.search(text)), "mech": bool(MECH_KW.search(text)),
        "weather": bool(WEATHER_KW.search(text)), "overtake": bool(OVERTAKE_KW.search(text)),
        "defend": bool(DEFEND_KW.search(text)), "drs": bool(DRS_KW.search(text)),
        "traffic": bool(TRAFFIC_KW.search(text)), "gap": bool(GAP_KW.search(text)),
        "positive": bool(POSITIVE_KW.search(text)),
    }

    # Primary event type
    primary = "information_only"
    if hits["pit"] and ("box" in text_lower or "pit" in text_lower):
        primary = "pit_call"
    elif hits["damage"]: primary = "damage_issue"
    elif hits["mech"]: primary = "mechanical_issue"
    elif hits["safety"]: primary = "safety"
    elif hits["weather"]: primary = "weather"
    elif hits["tire"] and not hits["pit"]: primary = "tire_strategy"
    elif hits["pace"]: primary = "pace_management"
    elif hits["overtake"]: primary = "overtaking"
    elif hits["defend"]: primary = "defending"
    elif hits["traffic"]: primary = "traffic"
    elif hits["positive"]: primary = "celebration"

    # Secondary labels
    secondary = []
    label_map = [("tire", "tire_strategy"), ("pit", "pit_call"), ("safety", "safety"),
                 ("pace", "pace_management"), ("damage", "damage_issue"),
                 ("mech", "mechanical_issue"), ("overtake", "overtaking"),
                 ("defend", "defending"), ("traffic", "traffic"), ("positive", "celebration")]
    for key, label in label_map:
        if hits[key] and label != primary and label not in secondary:
            secondary.append(label)

    # Action type
    action_required = True
    if primary == "pit_call" and "box" in text_lower: action_type = "pit_now"
    elif primary == "pit_call": action_type = "pit_soon"
    elif "stay out" in text_lower: action_type = "stay_out"
    elif "push" in text_lower or "attack" in text_lower: action_type = "push"
    elif "conserve" in text_lower or "save" in text_lower or "lift and coast" in text_lower: action_type = "conserve"
    elif hits["tire"] and ("manage" in text_lower or "look after" in text_lower): action_type = "manage_tires"
    elif hits["defend"]: action_type = "defend"
    elif hits["overtake"]: action_type = "overtake"
    elif hits["damage"] or hits["mech"]: action_type = "report_issue"
    elif primary in ("information_only", "celebration", "safety"):
        action_type = "acknowledge_info"; action_required = False
    else:
        action_type = "unknown"; action_required = False

    # Urgency
    if primary in ("pit_call", "damage_issue", "safety") or "now" in text_lower: urgency = "high"
    elif primary in ("mechanical_issue", "pace_management", "tire_strategy", "defending"): urgency = "medium"
    else: urgency = "low"

    # Sentiment
    if hits["positive"]: sentiment = "positive"
    elif hits["damage"] or hits["mech"]: sentiment = "negative"
    elif urgency == "high": sentiment = "urgent"
    else: sentiment = "neutral"

    # Quality & confidence
    word_count = len(text.split())
    quality = "low" if word_count < 3 else "medium" if word_count < 8 else "high"
    keyword_hits = sum(1 for v in hits.values() if v)
    if keyword_hits >= 2 and quality == "high": confidence = 0.85
    elif keyword_hits >= 1 and quality != "low": confidence = 0.7
    elif keyword_hits >= 1: confidence = 0.5
    else: confidence = 0.35

    # Evidence phrases
    evidence = []
    for pattern in ALL_PATTERNS:
        for match in pattern.finditer(text):
            start, end = max(0, match.start() - 15), min(len(text), match.end() + 15)
            snippet = text[start:end].strip()
            if snippet and snippet not in evidence:
                evidence.append(snippet)

    # Car issue
    if hits["damage"]:
        issue_type = "wing_damage" if "wing" in text_lower else "floor_damage" if "floor" in text_lower else "unknown"
        severity = "moderate"
    elif hits["mech"]:
        for kw, it in [("engine", "engine"), ("power unit", "engine"), ("brake", "brakes"),
                       ("gearbox", "gearbox"), ("battery", "battery"), ("ers", "battery"),
                       ("overheat", "overheating"), ("temperature", "overheating"), ("steering", "steering")]:
            if kw in text_lower: issue_type = it; break
        else: issue_type = "unknown"
        severity = "minor"
    else:
        issue_type = "none"; severity = "none"

    # Summary
    if primary == "pit_call": summary = "Pit call communicated."
    elif primary == "celebration": summary = "Positive message or celebration."
    elif primary == "information_only" and not secondary: summary = "General information exchanged."
    else:
        parts = [primary.replace('_', ' ').title()] + [s.replace('_', ' ') for s in secondary[:2]]
        summary = f"{'. '.join(parts)} related communication."

    return {
        "date": record.get("date"),
        "driver_number": int(record["driver_number"]) if pd.notna(record.get("driver_number")) else None,
        "meeting_key": int(record["meeting_key"]) if pd.notna(record.get("meeting_key")) else None,
        "session_key": int(record["session_key"]) if pd.notna(record.get("session_key")) else None,
        "recording_url": record.get("recording_url"),
        "transcript_cleaned": text,
        "transcript_quality": quality,
        "summary": summary,
        "primary_event_type": primary,
        "secondary_event_types": secondary,
        "speaker_direction": "unknown",
        "urgency_level": urgency,
        "sentiment_tone": sentiment,
        "action_required": action_required,
        "action_type": action_type,
        "strategy_signal": {
            "pit_related": hits["pit"], "tire_related": hits["tire"],
            "fuel_saving": "fuel" in text_lower or "lift and coast" in text_lower,
            "pace_change": hits["pace"], "weather_related": hits["weather"],
            "safety_related": hits["safety"],
        },
        "car_issue_signal": {"has_issue": hits["damage"] or hits["mech"], "issue_type": issue_type, "severity": severity},
        "racecraft_signal": {
            "traffic_mentioned": hits["traffic"], "overtake_mentioned": hits["overtake"],
            "defend_mentioned": hits["defend"], "drs_mentioned": hits["drs"],
            "gap_management_mentioned": hits["gap"],
        },
        "confidence": confidence,
        "evidence_phrases": evidence[:5],
        "notes": "" if confidence >= 0.7 else "Low confidence — transcript may be noisy or too short for reliable classification.",
    }

print("Classification functions loaded.")

Classification functions loaded.


## 4. Transcribe & Classify All Race Recordings

In [ ]:
from tqdm.notebook import tqdm
import time

events = []
failed = 0
total = len(race_radio)

pbar = tqdm(race_radio.iterrows(), total=total, desc="Transcribing", unit="rec",
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}] ✓{postfix}")
pbar.set_postfix(done=0, failed=0)

for i, (_, row) in enumerate(pbar):
    url = row["recording_url"]

    # Download
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
    except Exception:
        failed += 1
        pbar.set_postfix(done=len(events), failed=failed)
        continue

    # Write to temp file and transcribe
    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as f:
        f.write(r.content)
        tmppath = f.name

    try:
        result = model.transcribe(tmppath)
        transcript = result["text"].strip()
    except Exception:
        failed += 1
        os.unlink(tmppath)
        pbar.set_postfix(done=len(events), failed=failed)
        continue

    os.unlink(tmppath)

    if not transcript:
        failed += 1
        pbar.set_postfix(done=len(events), failed=failed)
        continue

    event = classify_radio(transcript, row.to_dict())
    events.append(event)
    pbar.set_postfix(done=len(events), failed=failed)

pbar.close()
print(f"\nDone! {len(events)} events from {total} recordings")
print(f"Failed/skipped: {failed}")

Transcribing:   0%|          | 0/6989 [00:00<?, ?rec/s] ✓


Done! 6662 events from 6989 recordings
Failed/skipped: 327


## 5. Save Results

In [ ]:
# Save as JSON
output_path = "data/team_radio_events.json"
with open(output_path, "w") as f:
    json.dump(events, f, indent=2)
print(f"Saved {len(events)} events to {output_path}")

# Also save a flat CSV for quick analysis
df_events = pd.json_normalize(events)
csv_path = "data/team_radio_events.csv"
df_events.to_csv(csv_path, index=False)
print(f"Saved CSV to {csv_path}")

Saved 6662 events to data/team_radio_events.json
Saved CSV to data/team_radio_events.csv


## 6. Summary Statistics

In [ ]:
df_events = pd.json_normalize(events)

print("EVENT TYPE DISTRIBUTION")
print(df_events["primary_event_type"].value_counts().to_string())

print("URGENCY DISTRIBUTION")
print(df_events["urgency_level"].value_counts().to_string())

print("ACTION TYPE DISTRIBUTION")
print(df_events["action_type"].value_counts().to_string())

print("CONFIDENCE DISTRIBUTION")
print(df_events["confidence"].describe().to_string())

print("STRATEGY SIGNALS")
for col in [c for c in df_events.columns if c.startswith("strategy_signal.")]:
    label = col.split(".")[1]
    print(f"  {label:<20} {df_events[col].sum():>5} occurrences")

print("SAMPLE EVENTS")
for _, row in df_events.head(5).iterrows():
    print(f"\n  Driver #{int(row['driver_number'])} | {row['primary_event_type']} | {row['urgency_level']}")
    print(f"  Transcript: \"{row['transcript_cleaned'][:100]}\"")
    print(f"  Summary: {row['summary']}")

EVENT TYPE DISTRIBUTION
primary_event_type
information_only    4095
pace_management      495
tire_strategy        422
celebration          414
safety               324
pit_call             197
weather              163
damage_issue         152
mechanical_issue     134
defending            127
overtaking           116
traffic               23
URGENCY DISTRIBUTION
urgency_level
low       4253
high      1449
medium     960
ACTION TYPE DISTRIBUTION
action_type
acknowledge_info    4745
unknown              780
push                 280
report_issue         247
defend               180
overtake             142
pit_soon             101
pit_now               96
conserve              35
manage_tires          28
stay_out              28
CONFIDENCE DISTRIBUTION
count    6662.000000
mean        0.518515
std         0.199336
min         0.350000
25%         0.350000
50%         0.350000
75%         0.700000
max         0.850000
STRATEGY SIGNALS
  pit_related            197 occurrences
  tire_related 